In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [3]:


# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100.0%
100.0%
100.0%
100.0%


In [20]:
batch_size = 64 # mini-batches. That one complete pass is called an epoch. The final mini-batch may contain fewer than 64 examples unless you specify:

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [24]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__() # __init__ sets up the model's layers. super().__init__() initializes the underlying PyTorch machinery.
        self.flatten = nn.Flatten() # [1, 28, 28] → [784] will be used below, forward()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512), # Takes 784 input values and produces 512 values.
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x) 
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device) # creates the model and moves its parameters to the selected device. The input tensors must be moved to the same device during training
print(model)

# There is no Softmax layer because, when using nn.CrossEntropyLoss(below), the loss function expects raw logits and applies the appropriate normalization internally.

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [25]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
# SGD vs Adam
# Adam uses the gradient plus running estimates of gradient magnitude and direction, so its update is more sophisticated than plain gradient descent. The overall idea remains the same: adjust parameters in a direction expected to reduce the loss.

In [26]:
# In a single training loop, the model makes predictions on the training dataset (fed to it in batches), and backpropagates the prediction error to adjust the model’s parameters.

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train() # Puts the model in training mode. This matters for layers such as dropout and batch normalization, whose behavior differs during training and evaluation.
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


# batch start at 0, one batch contains 64 sets
# - batch % 100 == 0 is true when batch is 0, 100, 200, and so on.
# - Therefore, progress is printed once every 100 batches.


In [36]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval() # Switches the model into evaluation mode.
    test_loss, correct = 0, 0
    with torch.no_grad(): # Disables gradient tracking. Evaluation does not update model weights, so gradients are unnecessary. This saves memory and computation. with torch.no_grad(): is a Python context manager. It temporarily changes how PyTorch behaves inside the indented block.
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item() # Here, 1 means “take the maximum across the class dimension,” dimension 1. If the largest value in the first row is at index 2, the model predicts class 2.
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

# explain correct
# 1. pred.argmax(1) selects the class with the highest score for each image.
# 2. == y compares predictions with the true labels.
# 3. .type(torch.float) converts True and False to 1.0 and 0.0.
# 4. .sum() counts how many predictions are correct.
# 5. .item() converts that count to a Python number.

In [37]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")


# explain process in mini-batches:

# flatten [64, 1, 28, 28] → [64, 784]
# last layer output logits.shape = [64, 10]
# [1.2, -0.4, 3.8, 0.1, ...]
# batch loss = (loss₁ + loss₂ + ... + loss₆₄) / 64

# loss.backward():
# gradient = ∂(batch loss) / ∂(parameter)
# model.linear_relu_stack[0].weight.grad [512, 784]

# optimizer.step():
# parameter_new = parameter_old - learning_rate × gradient

Epoch 1
-------------------------------
loss: 0.772177  [   64/60000]
loss: 0.858136  [ 6464/60000]
loss: 0.625178  [12864/60000]
loss: 0.849540  [19264/60000]
loss: 0.731976  [25664/60000]
loss: 0.730834  [32064/60000]
loss: 0.813268  [38464/60000]
loss: 0.794333  [44864/60000]
loss: 0.809341  [51264/60000]
loss: 0.765276  [57664/60000]
Test Error: 
 Accuracy: 71.4%, Avg loss: 0.774579 

Epoch 2
-------------------------------
loss: 0.772177  [   64/60000]
loss: 0.858136  [ 6464/60000]
loss: 0.625178  [12864/60000]
loss: 0.849540  [19264/60000]
loss: 0.731976  [25664/60000]
loss: 0.730834  [32064/60000]
loss: 0.813268  [38464/60000]
loss: 0.794333  [44864/60000]
loss: 0.809341  [51264/60000]
loss: 0.765276  [57664/60000]
Test Error: 
 Accuracy: 71.4%, Avg loss: 0.774579 

Epoch 3
-------------------------------
loss: 0.772177  [   64/60000]
loss: 0.858136  [ 6464/60000]
loss: 0.625178  [12864/60000]
loss: 0.849540  [19264/60000]
loss: 0.731976  [25664/60000]
loss: 0.730834  [32064/600

In [38]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [39]:
# The process for loading a model includes re-creating the model structure and loading the state dictionary into it.
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [53]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    print(pred)
    print(pred.shape)
    predicted, actual = classes[pred[0].argmax(0)], classes[y] # same as pred.argmax(1)
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

tensor([[-3.1918, -4.3763, -1.6961, -2.8412, -1.4533,  5.0419, -1.3770,  4.6457,
          2.7087,  5.3336]], device='cuda:0')
torch.Size([1, 10])
Predicted: "Ankle boot", Actual: "Ankle boot"
